# ML Firewall — Model Training & Evaluation

This notebook trains a **Random Forest binary classifier** to distinguish malicious from benign network traffic windows, evaluates its performance, and produces visualisations suitable for dissertation figures.

**Sections**
1. Imports & configuration
2. Load & clean dataset files
3. Exploratory data analysis (EDA)
4. Train / test split
5. Model training
6. Evaluation — classification report & confusion matrix
7. Threshold sensitivity analysis
8. Feature importance
9. Probability distribution analysis
10. False positive deep-dive


In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. Imports & configuration
# ─────────────────────────────────────────────────────────────────────────────
import glob
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, roc_curve, auc, precision_recall_curve
)
from sklearn.model_selection import GroupShuffleSplit


In [2]:
# ── Matplotlib style ──────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})
PALETTE = {"benign": "#1D9E75", "malicious": "#D85A30"}

# ── Feature list (must match capture_traffic.py / firewall.py) ────────────────
FEATURES = [
    "dest_port",       "window_duration",
    "fwd_packet_rate", "fwd_byte_rate",
    "bwd_packet_rate", "bwd_byte_rate",
    "pkt_len_mean",    "pkt_len_std",
    "pkt_len_min",     "pkt_len_max",
    "syn_ratio",       "fin_ratio",   "ack_ratio",
    "rst_ratio",       "psh_ratio",
    "fwd_bwd_ratio",   "byte_ratio",
    "iat_mean",        "iat_std",     "iat_min",
]

# ── Model hyperparameters ─────────────────────────────────────────────────────
RF_PARAMS = dict(
    n_estimators=200,
    max_depth=10,
    min_samples_split=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

# Thresholds to sweep during sensitivity analysis
THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

print("Configuration loaded.")
print(f"  Features : {len(FEATURES)}")
print(f"  RF params: {RF_PARAMS}")


Configuration loaded.
  Features : 20
  RF params: {'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 10, 'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1}


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. Load & clean dataset files
# ─────────────────────────────────────────────────────────────────────────────

files = sorted(glob.glob("dataset_*.csv"))
if not files:
    raise FileNotFoundError("No dataset_*.csv files found in the current directory.")

print(f"Found {len(files)} dataset file(s):")
for f in files:
    print(f"  {f}")

# Load each file, drop rows with any NaN, and tag with the source filename
dfs = []
for f in files:
    df = pd.read_csv(f)
    before = len(df)
    df.dropna(inplace=True)
    df["source_file"] = f
    dfs.append(df)
    print(f"  {f}: {before} rows → {len(df)} after dropna")

full_df = pd.concat(dfs, ignore_index=True)

# Strip window suffix (_w0, _w1 …) to get a stable base flow identifier.
# This is used later to prevent windows from the same flow leaking across
# the train/test boundary.
full_df["base_flow_id"] = full_df["flow_id"].str.replace(r"_w\d+$", "", regex=True)

# Drop exact feature-level duplicates (can arise from overlapping capture sessions)
before_dedup = len(full_df)
full_df = full_df.drop_duplicates(subset=FEATURES).reset_index(drop=True)

print(f"\nTotal rows  : {before_dedup}")
print(f"After dedup : {len(full_df)}  ({before_dedup - len(full_df)} removed)")
print(f"Unique windows (flow_id)    : {full_df['flow_id'].nunique()}")
print(f"Unique flows (base_flow_id) : {full_df['base_flow_id'].nunique()}")
print(f"\nLabel distribution:")
print(full_df["label"].value_counts())


FileNotFoundError: No dataset_*.csv files found in the current directory.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. Exploratory data analysis (EDA)
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# ── 3a. Class balance bar chart ───────────────────────────────────────────────
counts = full_df["label"].value_counts()
ax = axes[0]
bars = ax.bar(counts.index, counts.values,
              color=[PALETTE[l] for l in counts.index], width=0.5)
ax.set_title("Class balance")
ax.set_ylabel("Windows")
ax.set_xlabel("Label")
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            f"{val:,}", ha="center", va="bottom", fontsize=10)

# ── 3b. Distribution of a key rate feature by class ──────────────────────────
ax = axes[1]
for label, grp in full_df.groupby("label"):
    ax.hist(
        np.clip(grp["fwd_packet_rate"], 0, grp["fwd_packet_rate"].quantile(0.99)),
        bins=60, alpha=0.6, label=label, color=PALETTE[label], density=True
    )
ax.set_title("Forward packet rate distribution")
ax.set_xlabel("Packets / second  (clipped at 99th pct)")
ax.set_ylabel("Density")
ax.legend()

# ── 3c. SYN ratio by class ───────────────────────────────────────────────────
ax = axes[2]
for label, grp in full_df.groupby("label"):
    ax.hist(grp["syn_ratio"], bins=40, alpha=0.6,
            label=label, color=PALETTE[label], density=True)
ax.set_title("SYN ratio distribution")
ax.set_xlabel("SYN ratio")
ax.set_ylabel("Density")
ax.legend()

plt.suptitle("EDA — key feature distributions by class", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("fig_eda.png", bbox_inches="tight")
plt.show()

# ── 3d. Feature correlation heatmap ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 9))
corr = full_df[FEATURES].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".1f", linewidths=0.4,
            cmap="RdYlGn", center=0, vmin=-1, vmax=1,
            annot_kws={"size": 7}, ax=ax)
ax.set_title("Feature correlation matrix", fontsize=12)
plt.tight_layout()
plt.savefig("fig_correlation.png", bbox_inches="tight")
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. Train / test split  (group-aware — no flow leakage)
# ─────────────────────────────────────────────────────────────────────────────
#
# GroupShuffleSplit ensures that ALL windows from a given base flow
# are assigned to either train OR test — never split across both.
# Without this, a model could effectively memorise flow-specific
# statistics and report inflated accuracy at evaluation time.

groups = full_df["base_flow_id"].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(full_df, groups=groups))

train_df = full_df.iloc[train_idx].sample(frac=1, random_state=42).reset_index(drop=True)
test_df  = full_df.iloc[test_idx].sample(frac=1, random_state=42).reset_index(drop=True)

# ── Leakage verification ──────────────────────────────────────────────────────
leaked = set(train_df["base_flow_id"]) & set(test_df["base_flow_id"])
assert len(leaked) == 0, f"Leakage detected: {leaked}"
print("Leakage check passed — 0 shared base flow IDs.")

print(f"\nTrain: {len(train_df):,} windows")
print(train_df["label"].value_counts().to_string())
print(f"\nTest:  {len(test_df):,} windows")
print(test_df["label"].value_counts().to_string())

# ── Build numpy arrays ────────────────────────────────────────────────────────
X_train = train_df[FEATURES].values
y_train = (train_df["label"] == "malicious").astype(int).values

X_test  = test_df[FEATURES].values
y_test  = (test_df["label"] == "malicious").astype(int).values

# Sanity: no NaN or Inf in feature matrices
for name, X in [("X_train", X_train), ("X_test", X_test)]:
    assert not np.isnan(X).any(), f"NaN found in {name}"
    assert not np.isinf(X).any(), f"Inf found in {name}"
print("NaN / Inf check passed.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. Model training
# ─────────────────────────────────────────────────────────────────────────────

model = RandomForestClassifier(**RF_PARAMS)
print("Training Random Forest …")
model.fit(X_train, y_train)
print("Done.")

# Persist the trained model and the ordered feature list.
# Saving feature names alongside the model is critical: scikit-learn does NOT
# validate column order at inference time, so a silent column mismatch would
# produce wrong predictions without raising any error.
joblib.dump(model,    "rf_model.pkl")
joblib.dump(FEATURES, "feature_names.pkl")
print("Model saved → rf_model.pkl")
print("Feature names saved → feature_names.pkl")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6. Evaluation — classification report & confusion matrix
# ─────────────────────────────────────────────────────────────────────────────

y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["benign", "malicious"]))

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

fpr_val = fp / (fp + tn) if (fp + tn) > 0 else 0
fnr_val = fn / (fn + tp) if (fn + tp) > 0 else 0

print(f"True Positives  (attacks caught) : {tp}")
print(f"True Negatives  (benign correct) : {tn}")
print(f"False Positives (benign blocked) : {fp}")
print(f"False Negatives (attacks missed) : {fn}")
print(f"\nFalse Positive Rate (FPR): {fpr_val:.4f}  [target < 0.05]")
print(f"False Negative Rate (FNR): {fnr_val:.4f}  [target < 0.10]")

# ── Confusion matrix heatmap ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted benign", "Predicted malicious"],
            yticklabels=["Actual benign", "Actual malicious"],
            linewidths=0.5, ax=ax, cbar=False)
ax.set_title("Confusion matrix (default threshold = 0.5)")

# ── ROC curve ────────────────────────────────────────────────────────────────
ax = axes[1]
fpr_curve, tpr_curve, _ = roc_curve(y_test, y_pred_prob)
roc_auc = auc(fpr_curve, tpr_curve)
ax.plot(fpr_curve, tpr_curve, color="#534AB7", lw=2,
        label=f"ROC (AUC = {roc_auc:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5, label="Random baseline")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC curve")
ax.legend(loc="lower right")

plt.suptitle("Model evaluation", fontsize=12)
plt.tight_layout()
plt.savefig("fig_evaluation.png", bbox_inches="tight")
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 7. Threshold sensitivity analysis
# ─────────────────────────────────────────────────────────────────────────────
#
# The firewall applies a smoothed probability threshold rather than the
# default 0.5 decision boundary. This section shows how FPR, FNR, and F1
# vary with the threshold so the best operating point can be chosen.

rows = []
for t in THRESHOLDS:
    y_t = (y_pred_prob >= t).astype(int)
    cm_t = confusion_matrix(y_test, y_t, labels=[0, 1])
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    rows.append({
        "Threshold": t,
        "FPR": fp_t / (fp_t + tn_t) if (fp_t + tn_t) > 0 else 0,
        "FNR": fn_t / (fn_t + tp_t) if (fn_t + tp_t) > 0 else 0,
        "F1":  f1_score(y_test, y_t, zero_division=0),
        "TP":  tp_t, "FP": fp_t, "FN": fn_t, "TN": tn_t,
    })

sens_df = pd.DataFrame(rows)
print(sens_df.to_string(index=False))

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sens_df["Threshold"], sens_df["FPR"], marker="o",
        color=PALETTE["benign"],    label="FPR (false positive rate)", lw=2)
ax.plot(sens_df["Threshold"], sens_df["FNR"], marker="s",
        color=PALETTE["malicious"], label="FNR (false negative rate)", lw=2)
ax.plot(sens_df["Threshold"], sens_df["F1"],  marker="^",
        color="#534AB7",             label="F1 score",                  lw=2)

# Mark the threshold used in the live firewall
LIVE_THRESHOLD = 0.7
ax.axvline(LIVE_THRESHOLD, color="gray", linestyle="--", lw=1.2,
           label=f"Live threshold ({LIVE_THRESHOLD})")

ax.set_xlabel("Decision threshold")
ax.set_ylabel("Rate / score")
ax.set_title("Threshold sensitivity — FPR, FNR, and F1")
ax.set_xticks(THRESHOLDS)
ax.set_ylim(-0.02, 1.05)
ax.legend()
plt.tight_layout()
plt.savefig("fig_threshold_sensitivity.png", bbox_inches="tight")
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 8. Feature importance
# ─────────────────────────────────────────────────────────────────────────────
#
# Random Forest computes mean decrease in impurity (Gini importance) for each
# feature. Features near the top most strongly discriminate between benign
# and malicious windows. During development these scores were used to
# diagnose bugs in the feature extraction pipeline (a feature that should be
# highly discriminative but ranks low often signals a construction error).

importance_df = pd.DataFrame({
    "Feature":    FEATURES,
    "Importance": model.feature_importances_,
}).sort_values("Importance", ascending=False).reset_index(drop=True)

print(importance_df.to_string(index=False))

# ── Horizontal bar chart ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))
colors = ["#534AB7" if i < 5 else "#9EA0C8" for i in range(len(importance_df))]
ax.barh(importance_df["Feature"][::-1],
        importance_df["Importance"][::-1],
        color=colors[::-1], height=0.6)
ax.set_xlabel("Mean decrease in impurity (Gini importance)")
ax.set_title("Random Forest — feature importances")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))

# Annotation on top-5
for i, row in importance_df.head(5).iterrows():
    rank = len(importance_df) - i
    ax.text(row["Importance"] + 0.001, rank - 1,
            f'{row["Importance"]:.3f}', va="center", fontsize=9)

plt.tight_layout()
plt.savefig("fig_feature_importance.png", bbox_inches="tight")
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 9. Probability distribution analysis
# ─────────────────────────────────────────────────────────────────────────────
#
# A well-calibrated model should push benign windows toward 0 and malicious
# windows toward 1, with a clear gap around the decision threshold.
# Predictions clustered near 0.5 indicate the model is uncertain and the
# threshold choice becomes fragile.

prob_benign    = y_pred_prob[y_test == 0]
prob_malicious = y_pred_prob[y_test == 1]

print("Predicted probability (malicious class):")
print(f"  Benign windows    — mean: {prob_benign.mean():.4f}   std: {prob_benign.std():.4f}")
print(f"  Malicious windows — mean: {prob_malicious.mean():.4f}   std: {prob_malicious.std():.4f}")
print(f"\nBenign windows predicted > 0.7 : {(prob_benign > 0.7).sum()} / {len(prob_benign)}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Overlapping histogram ─────────────────────────────────────────────────────
ax = axes[0]
ax.hist(prob_benign,    bins=50, alpha=0.6, color=PALETTE["benign"],
        density=True, label=f"Benign (n={len(prob_benign):,})")
ax.hist(prob_malicious, bins=50, alpha=0.6, color=PALETTE["malicious"],
        density=True, label=f"Malicious (n={len(prob_malicious):,})")
ax.axvline(0.7, color="gray", linestyle="--", lw=1.2, label="Live threshold (0.7)")
ax.set_xlabel("Predicted probability (malicious class)")
ax.set_ylabel("Density")
ax.set_title("Predicted probability distribution by true class")
ax.legend()

# ── Precision-Recall curve ────────────────────────────────────────────────────
ax = axes[1]
precision, recall, pr_thresholds = precision_recall_curve(y_test, y_pred_prob)
pr_auc = auc(recall, precision)
ax.plot(recall, precision, color="#D85A30", lw=2,
        label=f"PR curve (AUC = {pr_auc:.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall curve")
ax.legend()

plt.suptitle("Probability calibration", fontsize=12)
plt.tight_layout()
plt.savefig("fig_probability_distribution.png", bbox_inches="tight")
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 10. False positive deep-dive
# ─────────────────────────────────────────────────────────────────────────────
#
# False positives (benign windows classified as malicious) are the most
# operationally costly error type — they block legitimate traffic.
# Examining which feature values appear in FP windows helps identify
# benign traffic patterns that resemble attacks, and informs whether
# to adjust the threshold or add more representative training data.

fp_mask = (y_test == 0) & (y_pred == 1)
fp_count = fp_mask.sum()
total_benign = (y_test == 0).sum()

print(f"False positives: {fp_count} / {total_benign} benign windows "
      f"({100 * fp_count / total_benign:.2f}%)")

if fp_count > 0:
    fp_df  = pd.DataFrame(X_test[fp_mask],  columns=FEATURES)
    all_b  = pd.DataFrame(X_test[y_test==0], columns=FEATURES)

    compare = pd.DataFrame({
        "All benign (mean)": all_b.mean(),
        "FP windows (mean)": fp_df.mean(),
        "Ratio (FP / benign)": (fp_df.mean() / all_b.mean().replace(0, np.nan)).round(2),
    })
    print("\nFeature means — false positive windows vs. all benign windows:")
    print(compare.to_string())

    # ── Bar chart: top differentiating features ───────────────────────────────
    ratio = compare["Ratio (FP / benign)"].dropna().sort_values(ascending=False)
    top_n = ratio.head(10)

    fig, ax = plt.subplots(figsize=(8, 4))
    top_n[::-1].plot.barh(ax=ax, color="#D85A30", alpha=0.8)
    ax.axvline(1.0, color="gray", linestyle="--", lw=1, label="No difference (ratio = 1)")
    ax.set_xlabel("FP mean / all-benign mean")
    ax.set_title("Top features elevated in false positive windows")
    ax.legend()
    plt.tight_layout()
    plt.savefig("fig_false_positives.png", bbox_inches="tight")
    plt.show()
else:
    print("No false positives — nothing to plot.")
